In [ ]:
%pipinstallgoogle-generativeaipandasopenpyxltqdm--quiet

In [ ]:
importos,json,time
importpandasaspd
fromtqdmimporttqdm
fromcollectionsimportCounter
importgoogle.generativeaiasgenai

print("Imports erfolgreich.")

In [ ]:
GEMINI_REDACTED_API_KEY

MODEL_NAME="gemini-2.5-flash-lite"
DELAY_BETWEEN_CALLS=0.1

INPUT_FILE="DDL_Binary_Step1.xlsx"
INPUT_SHEET="DDL_relevant"
CHECKPOINT_CSV="DDL_Stage_Checkpoint_v4.csv"
OUTPUT_EXCEL="DDL_Stage_Classification_v4.xlsx"

importgoogle.generativeaiasgenai
genai.configure(api_key=GEMINI_API_KEY)
model=genai.GenerativeModel(MODEL_NAME)

print(f"Modell    : {MODEL_NAME}")
print(f"Input     : {INPUT_FILE} (Sheet: {INPUT_SHEET})")
print(f"Output    : {OUTPUT_EXCEL}")

In [ ]:
STAGE_DEFINITION=(
"The Drug Development Lifecycle (DDL) consists of four stages:

"
"Stage 1 — Discovery & Development:
"
"   Hypothesis-driven research; aims at identifying and characterising drug targets; "
"encompasses assay development, high-throughput compound screening, hit-to-lead "
"optimisation, early pharmacological profiling, and preliminary safety assessments "
"conducted outside formal GLP frameworks.

"
"Stage 2 — Preclinical Research:
"
"   GLP-compliant laboratory studies, both in vitro and in vivo, evaluating the safety, "
"toxicity, absorption, distribution, metabolism, and excretion properties of drug "
"candidates prior to first-in-human administration; includes IND-enabling studies and "
"preparation of data packages for Phase I submission.

"
"Stage 3 — Clinical Research:
"
"   Human clinical trials across Phase I, II, and III evaluating safety, tolerability, "
"dosing, and therapeutic efficacy; encompasses trial design, patient recruitment, "
"clinical data monitoring, biostatistics, pharmacovigilance, and regulatory submissions "
"tied to the conduct of trials.

"
"Stage 4 — Regulatory Review:
"
"   Preparation and submission of a New Drug Application (NDA) or equivalent dossier "
"for marketing authorisation; includes benefit–risk assessment, statistical analysis "
"for submission, product labelling, manufacturing quality control, and regulatory "
"affairs activities at the point of agency review."
)

ASSIGNMENT_RULES=(
"Assignment rules:\n"
"1. Assign probabilities (0-100, summing to 100) for each stage.\n"
"2. Assign the task to the stage with the HIGHEST probability (dominant stage).\n"
"3. If a task appears relevant to multiple stages, assign it to the stage that "
"reflects its PRIMARY FUNCTIONAL OBJECTIVE.\n"
"4. Provide a one-sentence justification explaining why this stage is dominant "
"relative to other possible stages.\n"
"5. This ensures mutually exclusive lifecycle aggregation and prevents artificial "
"inflation of stage-level exposure."
)

FEW_SHOT_STAGES=(
"Examples:\n\n"
"Occupation: Biochemists and Biophysicists\n"
'Task: "Screen compound libraries to identify drug candidate hits."\n'
'{"prob_1": 85, "prob_2": 10, "prob_3": 3, "prob_4": 2, "dominant_stage": 1,'
' "reason": "Hit screening is a core Discovery & Development activity, not a GLP study."}\n\n'
"Occupation: Medical Scientists\n"
'Task: "Conduct GLP toxicology studies in rodent models for IND submission."\n'
'{"prob_1": 5, "prob_2": 90, "prob_3": 3, "prob_4": 2, "dominant_stage": 2,'
' "reason": "GLP toxicology for IND filing is the defining Preclinical activity, not discovery or clinical work."}\n\n'
"Occupation: Clinical Research Coordinators\n"
'Task: "Recruit and monitor patients in Phase II clinical trials."\n'
'{"prob_1": 2, "prob_2": 3, "prob_3": 93, "prob_4": 2, "dominant_stage": 3,'
' "reason": "Patient recruitment for Phase II is exclusively a Clinical Research task."}\n\n'
"Occupation: Regulatory Affairs Specialists\n"
'Task: "Prepare and submit NDA dossier to the FDA."\n'
'{"prob_1": 2, "prob_2": 3, "prob_3": 10, "prob_4": 85, "dominant_stage": 4,'
' "reason": "NDA preparation is the primary Regulatory Review activity, not clinical trial conduct."}\n\n'
"Occupation: Biostatisticians\n"
'Task: "Develop statistical models for analysing clinical trial and regulatory submission data."\n'
'{"prob_1": 5, "prob_2": 5, "prob_3": 55, "prob_4": 35, "dominant_stage": 3,'
' "reason": "Although spanning both Clinical and Regulatory contexts, the primary function is trial data analysis."}\n\n'
"Occupation: Medical Scientists\n"

)

SYSTEM_PROMPT_STAGE=(
"You are an expert in pharmaceutical drug development and occupational task analysis.\n\n"
"For the following occupational task, assign lifecycle stage probabilities and "
"identify the dominant stage.\n\n"
+STAGE_DEFINITION
+"\n\n"+ASSIGNMENT_RULES
+"\n\n"+FEW_SHOT_STAGES
+"\n\nRespond ONLY with valid JSON, no other text:\n"
'{"prob_1": <0-100>, "prob_2": <0-100>, "prob_3": <0-100>, "prob_4": <0-100>,'
' "dominant_stage": <1-4>, "reason": "<one sentence why this stage dominates>"}'
)

defbuild_prompt(occupation,task):
    returnf'Now classify:\nOccupation: "{occupation}"\nTask: "{task}"'

print(f"Prompt: {len(SYSTEM_PROMPT_STAGE)} Zeichen")
print("Prompt bereit.")

In [ ]:
MAX_RETRIES=3
RETRY_DELAY=10

defparse_stage_response(raw):
    try:
        clean=raw.strip()
if"```"inclean:
            parts=clean.split("```")
clean=parts[1]iflen(parts)>1elseparts[0]
ifclean.startswith("json"):
                clean=clean[4:]
start=clean.find("{")
end=clean.rfind("}")+1
ifstart!=-1andend>start:
            clean=clean[start:end]
parsed=json.loads(clean.strip())
prob_1=int(parsed.get("prob_1",0))
prob_2=int(parsed.get("prob_2",0))
prob_3=int(parsed.get("prob_3",0))
prob_4=int(parsed.get("prob_4",0))
dominant=int(parsed.get("dominant_stage",-1))
ifdominantnotin(1,2,3,4):

            probs=[prob_1,prob_2,prob_3,prob_4]
dominant=probs.index(max(probs))+1
return{
"prob_1":prob_1,"prob_2":prob_2,
"prob_3":prob_3,"prob_4":prob_4,
"dominant_stage":dominant,
"reason":parsed.get("reason","").strip()
}
exceptExceptionase:
        return{
"prob_1":0,"prob_2":0,"prob_3":0,"prob_4":0,
"dominant_stage":-1,
"reason":f"Parse error: {str(e)[:80]}"
}


defquery_gemini_stage(occupation,task):
    full_prompt=SYSTEM_PROMPT_STAGE+"\n\n"+build_prompt(occupation,task)
forattemptinrange(1,MAX_RETRIES+1):
        try:
            resp=model.generate_content(
full_prompt,
generation_config=genai.types.GenerationConfig(
temperature=0,
max_output_tokens=200,
)
)
result=parse_stage_response(resp.text)
ifresult["dominant_stage"]!=-1:
                returnresult
ifattempt<MAX_RETRIES:
                time.sleep(RETRY_DELAY)
exceptExceptionase:
            err=str(e)
if"429"inerror"quota"inerr.lower():
                time.sleep(60+attempt*15)
elifattempt<MAX_RETRIES:
                time.sleep(RETRY_DELAY)
return{
"prob_1":0,"prob_2":0,"prob_3":0,"prob_4":0,
"dominant_stage":-1,"reason":"Alle Versuche fehlgeschlagen"
}


STAGE_NAMES={
1:"Discovery & Development",
2:"Preclinical Research",
3:"Clinical Research",
4:"Regulatory Review",
}

print("Funktionen bereit.")

In [ ]:
test_cases=[
("Biochemists","Screen compound libraries to identify molecular hits.",1),
("Medical Scientists","Conduct GLP-compliant toxicology studies in rodent models for IND filing.",2),
("Clinical Research Coordinators","Recruit and monitor patients in Phase III clinical trials.",3),
("Regulatory Affairs Specialists","Prepare and submit the NDA dossier to the FDA.",4),
]

print("="*70)
all_ok=True
forocc,task,expectedintest_cases:
    r=query_gemini_stage(occ,task)
time.sleep(DELAY_BETWEEN_CALLS)
ok=r["dominant_stage"]==expected
ifnotok:
        all_ok=False
status="OK"ifokelsef"FALSCH (erwartet {expected})"
print(f"Stage {r['dominant_stage']} — {STAGE_NAMES.get(r['dominant_stage'],'?'):<30} [{status}]")
print(f"  Probs: S1={r['prob_1']}% S2={r['prob_2']}% S3={r['prob_3']}% S4={r['prob_4']}%")
print(f"  Reason: {r['reason'][:80]}")
print("-"*70)
print()
print("Bereit fuer vollen Lauf!"ifall_okelse"Prompt ueberpruefen!")

In [ ]:
df=pd.read_excel(INPUT_FILE,sheet_name=INPUT_SHEET)
df["Task ID"]=df["Task ID"].astype(str)

print(f"Geladen: {len(df):,} Tasks aus {df['Title'].nunique()} Occupations")
print(f"Spalten: {list(df.columns)}")
df.head(3)

In [ ]:
ifos.path.exists(CHECKPOINT_CSV):
    done_df=pd.read_csv(CHECKPOINT_CSV,dtype={"Task ID":str})
done_ids=set(done_df["Task ID"].tolist())
print(f"Resume: {len(done_ids):,} Tasks bereits klassifiziert.")
else:
    done_df=pd.DataFrame()
done_ids=set()
print("Starte neu.")

remaining=df[~df["Task ID"].isin(done_ids)].copy()
print(f"Verbleibend: {len(remaining):,} Tasks")

buffer=[]

for_,rowintqdm(remaining.iterrows(),total=len(remaining),desc="Stage-Klassifikation"):
    r=query_gemini_stage(str(row["Title"]),str(row["Task"]))
buffer.append({
"Task ID":str(row["Task ID"]),
"prob_1":r["prob_1"],
"prob_2":r["prob_2"],
"prob_3":r["prob_3"],
"prob_4":r["prob_4"],
"dominant_stage":r["dominant_stage"],
"stage_name":STAGE_NAMES.get(r["dominant_stage"],"Error"),
"reason_stage":r["reason"],
})

time.sleep(DELAY_BETWEEN_CALLS)

iflen(buffer)%50==0:
        snap=pd.concat([done_df,pd.DataFrame(buffer)],ignore_index=True)
snap.drop_duplicates(subset="Task ID",keep="last").to_csv(CHECKPOINT_CSV,index=False)
print(f"  Checkpoint: {len(snap):,} Tasks gespeichert.")

final_stages=pd.concat([done_df,pd.DataFrame(buffer)],ignore_index=True)ifbufferelsedone_df
final_stages.drop_duplicates(subset="Task ID",keep="last").to_csv(CHECKPOINT_CSV,index=False)
print(f"\nFertig! {len(final_stages):,} Tasks klassifiziert.")

In [ ]:

stage_df=pd.read_csv(CHECKPOINT_CSV,dtype={"Task ID":str})


merged=df.merge(
stage_df[["Task ID","prob_1","prob_2","prob_3","prob_4",
"dominant_stage","stage_name","reason_stage"]],
on="Task ID",how="left"
)
merged["dominant_stage"]=merged["dominant_stage"].fillna(-1).astype(int)
merged["stage_name"]=merged["dominant_stage"].map(STAGE_NAMES).fillna("Error")

out_cols=[
"O*NET-SOC Code","Title","Task ID","Task","Task Type",
"dominant_stage","stage_name",
"prob_1","prob_2","prob_3","prob_4",
"reason_stage",
"D_final","D_gemini","D_deepseek","D_mistral32",
]
out_cols=[cforcinout_colsifcinmerged.columns]
out=merged[out_cols].sort_values(["dominant_stage","Title"]).reset_index(drop=True)

print("Verteilung der 4 Stages:")
print("-"*55)
forsn,nameinSTAGE_NAMES.items():
    n=(out["dominant_stage"]==sn).sum()
pct=100*n/len(out)iflen(out)>0else0
bar="█"*int(pct/2)
print(f"  Stage {sn} — {name:<30}: {n:3d} ({pct:4.1f}%)  {bar}")
print(f"  Total: {len(out)}")

In [ ]:
fromopenpyxl.stylesimportFont,PatternFill,Alignment
fromopenpyxl.utilsimportget_column_letter
fromopenpyxlimportload_workbook

STAGE_COLORS={
1:"D6E4F0",
2:"D5F5E3",
3:"FEF9E7",
4:"FADBD8",
}

defstyle_sheet(ws):
    forcellinws[1]:
        cell.font=Font(bold=True,color="FFFFFF",name="Arial",size=10)
cell.fill=PatternFill("solid",fgColor="2E4057")
cell.alignment=Alignment(horizontal="center",wrap_text=True)
headers=[cell.valueforcellinws[1]]
stage_idx=headers.index("dominant_stage")+1if"dominant_stage"inheaderselseNone
forrowinws.iter_rows(min_row=2):
        stage_val=row[stage_idx-1].valueifstage_idxelseNone
fill_color=STAGE_COLORS.get(stage_val,"FFFFFF")
forcellinrow:
            cell.fill=PatternFill("solid",fgColor=fill_color)
cell.font=Font(name="Arial",size=9)
cell.alignment=Alignment(wrap_text=True,vertical="top")
fori,colinenumerate(ws.columns,1):
        header=ws.cell(1,i).valueor""
ifheader=="Task":
            ws.column_dimensions[get_column_letter(i)].width=60
elif"reason"instr(header).lower():
            ws.column_dimensions[get_column_letter(i)].width=50
elifheaderin("Title","stage_name"):
            ws.column_dimensions[get_column_letter(i)].width=28
elif"prob_"instr(header):
            ws.column_dimensions[get_column_letter(i)].width=10
else:
            ws.column_dimensions[get_column_letter(i)].width=14
ws.freeze_panes="A2"

withpd.ExcelWriter(OUTPUT_EXCEL,engine="openpyxl")aswriter:
    out.to_excel(writer,sheet_name="Stage_Classification",index=False)

wb=load_workbook(OUTPUT_EXCEL)
style_sheet(wb["Stage_Classification"])
wb.save(OUTPUT_EXCEL)

print(f"Exportiert: {OUTPUT_EXCEL}")
print(f"  Stage_Classification: {len(out):,} Tasks")

In [ ]:
print("="*55)
print("DDL STAGE KLASSIFIKATION — Single Stage Assignment")
print("="*55)
forsn,nameinSTAGE_NAMES.items():
    n=(out["dominant_stage"]==sn).sum()
pct=100*n/len(out)iflen(out)>0else0
bar="█"*int(pct/2)
print(f"  {sn} — {name:<30}: {n:3d} ({pct:4.1f}%)  {bar}")
print("-"*55)
print(f"  Total: {len(out)} Tasks")
print()
print("Durchschnittliche Konfidenz (Wahrscheinlichkeit dominante Stage):")
conf=[]
for_,rowinout.iterrows():
    ds=row["dominant_stage"]
ifdsin(1,2,3,4):
        conf.append(row[f"prob_{ds}"])
ifconf:
    print(f"  Mean: {sum(conf)/len(conf):.1f}%")
print(f"  Min:  {min(conf):.0f}%")
print(f"  Max:  {max(conf):.0f}%")